# Experiment

In [1]:
import pyterrier as pt
import pandas as pd
import os
from pathlib import Path

In [2]:
dataset_name = "irds:cord19/trec-covid"
dataset = pt.get_dataset(dataset_name)

## Index Creation/Loading

In [3]:
index_dir_path = Path("index/trec_covid_index_default").resolve()
index_dir = str(index_dir_path)
index_properties_file = os.path.join(index_dir, "data.properties")

In [4]:
os.makedirs(index_dir, exist_ok=True)

In [9]:
def trec_covid_corpus_generator():
    for doc in dataset.get_corpus_iter():
        title = str(doc.get('title', ''))
        abstract = str(doc.get('abstract', ''))

        yield {
            'docno': doc['docno'],
            'text': title + " " + abstract
        }

In [10]:
if not pt.started():
    pt.init()

/var/folders/vs/5jv9s671375b6_36f8xp40s00000gn/T/ipykernel_2063/3057724015.py:1: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():
The operation couldn’t be completed. Unable to locate a Java Runtime.
Please visit http://www.java.com for information on installing Java.



Exception: You must install Java for Mac OSX

In [7]:
if os.path.exists(index_properties_file):
    print(f"Index found at {index_dir}. Loading it...")
    index_ref = pt.IndexRef.of(index_properties_file)
else:
    print(f"Index not found at {index_dir}. Building it now (this may take a few minutes)...")
    
    indexer = pt.IterDictIndexer(index_dir, blocks=True, meta={'docno': 50})
    index_ref = indexer.index(trec_covid_corpus_generator())
    print("Index built successfully.")

Index not found at /Users/nataliemladenova/Desktop/Master/Block3/IR/Group Project/IR_Project/src/index/trec_covid_index_default. Building it now (this may take a few minutes)...


The operation couldn’t be completed. Unable to locate a Java Runtime.
Please visit http://www.java.com for information on installing Java.



Exception: You must install Java for Mac OSX

## Different Query Verbosity Levels

In [32]:
topics_data = []
for query in dataset.irds_ref().queries_iter():
    topics_data.append({
        'qid': query.query_id,
        'title': query.title,
        'description': query.description,
        'narrative': query.narrative
    })
df_all_topics = pd.DataFrame(topics_data)

In [33]:
# TITLE Queries
topics_title = df_all_topics[['qid', 'title']].copy()
topics_title = topics_title.rename(columns={'title': 'query'})

# DESCRIPTION Queries
topics_desc = df_all_topics[['qid', 'description']].copy()
topics_desc = topics_desc.rename(columns={'description': 'query'})

# NARRATIVE Queries
topics_narr = df_all_topics[['qid', 'narrative']].copy()
topics_narr = topics_narr.rename(columns={'narrative': 'query'})

## Models

In [34]:
bm25 = pt.terrier.Retriever(index_ref, wmodel="BM25")
ql_dir = pt.terrier.Retriever(index_ref, wmodel="DirichletLM")
ql_jm = pt.terrier.Retriever(index_ref, wmodel="Hiemstra_LM")
rm3_pipe = bm25 >> pt.rewrite.RM3(index_ref) >> bm25

In [35]:
models = [bm25, ql_dir, ql_jm, rm3_pipe]
model_names = ["BM25", "QL (Dir)", "QL (JM)", "BM25 + RM3"]

## Experiment

In [36]:
eval_metrics = [
    "ndcg_cut_10",
    "P_5",
    "recip_rank",
    "recall_100",
    "map"
]
query_variants = {
    "Title": topics_title,
    "Description": topics_desc,
    "Narrative": topics_narr
}

In [37]:
for variant_name, topics_df in query_variants.items():
    print(f"==================================================")
    print(f"Results for {variant_name.upper()} Queries:")
    print(f"==================================================")
    
    experiment_results = pt.Experiment(
        retr_systems=models,
        topics=topics_df,
        qrels=dataset.get_qrels(),
        eval_metrics=eval_metrics,
        names=model_names,
        baseline=0
    )
    
    display(experiment_results)
    print("\n")

Results for TITLE Queries:


,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.207121,0.807094,0.672,0.106150,0.601847,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.175731,0.684478,0.540,0.094298,0.444853,14.0,36.0,3.445424e-04,5.0,...,0.011665,8.0,23.0,0.000784,16.0,33.0,5.811970e-03,8.0,40.0,4.739597e-07
2,QL (JM),0.128675,0.657621,0.452,0.075799,0.389103,2.0,48.0,2.471805e-09,7.0,...,0.018710,5.0,30.0,0.000003,5.0,44.0,4.825223e-10,8.0,40.0,1.121550e-08
3,BM25 + RM3,0.215513,0.777866,0.684,0.109137,0.580049,30.0,20.0,1.205768e-01,4.0,...,0.202878,13.0,12.0,0.695299,29.0,18.0,2.391932e-01,21.0,25.0,2.834572e-01




Results for DESCRIPTION Queries:


,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.222260,0.874500,0.716,0.112934,0.623256,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.185737,0.757972,0.592,0.101559,0.505937,9.0,41.0,2.067988e-05,7.0,...,0.037531,7.0,23.0,0.001941,16.0,29.0,2.971897e-03,15.0,35.0,0.000169
2,QL (JM),0.125899,0.715361,0.528,0.078242,0.444361,1.0,49.0,1.163376e-11,5.0,...,0.006109,6.0,30.0,0.000172,5.0,44.0,2.482803e-09,12.0,38.0,0.000012
3,BM25 + RM3,0.237755,0.842557,0.752,0.117835,0.655691,32.0,18.0,4.463815e-02,2.0,...,0.038771,13.0,10.0,0.192114,27.0,16.0,2.018099e-01,26.0,20.0,0.126493




Results for NARRATIVE Queries:


,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.157843,0.718762,0.608,0.089073,0.518681,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.105462,0.556934,0.452,0.069563,0.347096,8.0,42.0,4.518420e-07,7.0,...,0.011115,10.0,26.0,0.003339,12.0,38.0,0.000012,11.0,35.0,0.000005
2,QL (JM),0.117736,0.638829,0.440,0.072307,0.388177,7.0,43.0,1.239431e-05,8.0,...,0.194593,6.0,25.0,0.000344,10.0,38.0,0.000231,11.0,33.0,0.000070
3,BM25 + RM3,0.175874,0.678626,0.560,0.090549,0.489098,19.0,31.0,1.503040e-01,4.0,...,0.101944,7.0,16.0,0.122430,17.0,30.0,0.703406,17.0,27.0,0.135629


## Experiment with Dense Retrieval (DPR)

Dense retrieval uses **pyterrier-dr** with the supported pipeline: `doc_encoder` → `indexer` to build `index/trec_covid_dense`, then `query_encoder` → `retriever` for evaluation (same **title + abstract** text as the sparse index). If the FlexIndex is missing, the next cell builds it automatically—expect a long first run on CPU.

Set `DPR_REBUILD = True` in the setup cell if you need a clean rebuild after swapping the sentence-transformers model.

In [11]:
# pyterrier-dr from PyPI tracks the documented indexing/retrieval API.
!pip install -q torch sentence-transformers faiss-cpu pyterrier-dr

In [ ]:
import sys
from pathlib import Path

# Notebook may run with cwd = repo/src or repo root
for _p in (Path.cwd(), Path.cwd() / "src"):
    if (_p / "dpr_dense.py").is_file():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break

from dpr_dense import make_dpr_pipeline

# Same MiniLM bi-encoder as before; index lives under src/index/ when the notebook cwd is src/.
DPR_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
DPR_REBUILD = False  # set True to wipe and rebuild the FlexIndex

dense_index_dir = Path("index/trec_covid_dense").resolve()
dpr_pipe = make_dpr_pipeline(
    dataset,
    dense_index_dir,
    DPR_MODEL,
    rebuild=DPR_REBUILD,
)
print("DPR pipeline ready (dense retrieval over title+abstract).")

/Users/nataliemladenova/Desktop/Master/Block3/IR/Group Project/IR_Project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6094.74it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Building dense FlexIndex (first run is slow on CPU; use GPU if available).


indexing: 43701dvec [08:45, 56.40dvec/s] 

In [39]:
all_models = [bm25, ql_dir, ql_jm, rm3_pipe, dpr_pipe]
all_model_names = ["BM25", "QL (Dir)", "QL (JM)", "BM25 + RM3", "DPR (all-MiniLM)"]

for variant_name, topics_df in query_variants.items():
    print(f"==================================================")
    print(f"Results for {variant_name.upper()} Queries:")
    print(f"==================================================")
    
    experiment_results = pt.Experiment(
        retr_systems=all_models,
        topics=topics_df,
        qrels=dataset.get_qrels(),
        eval_metrics=eval_metrics,
        names=all_model_names,
        baseline=0 
    )
    
    display(experiment_results)
    print("\n")

Results for TITLE Queries:


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.207121,0.807094,0.672,0.106150,0.601847,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.175731,0.684478,0.540,0.094298,0.444853,14.0,36.0,3.445424e-04,5.0,...,0.011665,8.0,23.0,7.844063e-04,16.0,33.0,5.811970e-03,8.0,40.0,4.739597e-07
2,QL (JM),0.128675,0.657621,0.452,0.075799,0.389103,2.0,48.0,2.471805e-09,7.0,...,0.018710,5.0,30.0,2.983346e-06,5.0,44.0,4.825223e-10,8.0,40.0,1.121550e-08
3,BM25 + RM3,0.215513,0.777866,0.684,0.109137,0.580049,30.0,20.0,1.205768e-01,4.0,...,0.202878,13.0,12.0,6.952988e-01,29.0,18.0,2.391932e-01,21.0,25.0,2.834572e-01
4,DPR (all-MiniLM),0.075122,0.563015,0.340,0.050606,0.299373,3.0,47.0,2.412138e-09,6.0,...,0.000482,3.0,34.0,1.268344e-08,5.0,43.0,1.140979e-08,6.0,43.0,2.527973e-09




Results for DESCRIPTION Queries:


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.222260,0.874500,0.716,0.112934,0.623256,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.185737,0.757972,0.592,0.101559,0.505937,9.0,41.0,2.067988e-05,7.0,...,0.037531,7.0,23.0,0.001941,16.0,29.0,2.971897e-03,15.0,35.0,0.000169
2,QL (JM),0.125899,0.715361,0.528,0.078242,0.444361,1.0,49.0,1.163376e-11,5.0,...,0.006109,6.0,30.0,0.000172,5.0,44.0,2.482803e-09,12.0,38.0,0.000012
3,BM25 + RM3,0.237755,0.842557,0.752,0.117835,0.655691,32.0,18.0,4.463815e-02,2.0,...,0.038771,13.0,10.0,0.192114,27.0,16.0,2.018099e-01,26.0,20.0,0.126493
4,DPR (all-MiniLM),0.119940,0.680775,0.524,0.075400,0.448718,6.0,44.0,2.223430e-09,5.0,...,0.004876,9.0,29.0,0.000935,8.0,41.0,1.290954e-06,12.0,38.0,0.000474




Results for NARRATIVE Queries:


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.157843,0.718762,0.608,0.089073,0.518681,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.105462,0.556934,0.452,0.069563,0.347096,8.0,42.0,4.518420e-07,7.0,...,0.011115,10.0,26.0,0.003339,12.0,38.0,0.000012,11.0,35.0,0.000005
2,QL (JM),0.117736,0.638829,0.440,0.072307,0.388177,7.0,43.0,1.239431e-05,8.0,...,0.194593,6.0,25.0,0.000344,10.0,38.0,0.000231,11.0,33.0,0.000070
3,BM25 + RM3,0.175874,0.678626,0.560,0.090549,0.489098,19.0,31.0,1.503040e-01,4.0,...,0.101944,7.0,16.0,0.122430,17.0,30.0,0.703406,17.0,27.0,0.135629
4,DPR (all-MiniLM),0.111529,0.542542,0.424,0.069849,0.368355,14.0,36.0,2.405376e-03,9.0,...,0.023023,9.0,28.0,0.003070,13.0,36.0,0.004678,11.0,35.0,0.000409


## Dense retrieval: `dpr_dense.py` and `dpr_finetune.py`

Baseline **DPR (MiniLM)** is created with **`make_dpr_pipeline`** in `dpr_dense.py` (see the setup cells earlier in this notebook): it builds or loads a **pyterrier-dr FlexIndex** over **title + abstract**, same text as BM25. Fine-tuning is **not** run inside the notebook here; use the **script** instead (below).

### `dpr_dense.py` (indexing + retrieval)

| Piece | Role |
|-------|------|
| **`iter_flex_docs(dataset)`** | Streams `{"docno", "text"}` for the indexer (pyterrier-dr expects **`docno`**). |
| **`make_dpr_pipeline(...)`** | If `pt_meta.json` is missing, runs `doc_encoder >> FlexIndex.indexer(overwrite)` over the corpus, then returns **`query_encoder >> retriever`** plus a small transform so rankings expose **`docno`** for `pt.Experiment`. |
| **`retriever_for_loaded_encoder(enc, index_dir)`** | After you swap in another `SBertBiEncoder` checkpoint, load the matching on-disk index and get the same retrieval transformer. |
| **`make_biodpr_pipeline`** | Same pattern for a **BioMed** encoder when a pre-built FlexIndex exists. |

### `dpr_finetune.py` (supervision from qrels)

| Piece | Role |
|-------|------|
| **`build_training_pairs(dataset, df_all_topics, qrels, verbosity=..., min_label=1)`** | For each positive qrel, pairs **topic text** (title / description / narrative, or **all** non-empty fields) with the **positive document** text. Used for contrastive training. |
| **`fit_sentence_transformer(pairs, base_model, output_dir, ...)`** | **MultipleNegativesRankingLoss** fine-tuning via sentence-transformers. |
| **CLI** | From `src/`: `python dpr_finetune.py --pairs-smoke` (tiny sanity check), `python dpr_finetune.py --train --verbosity all` (full run; use **GPU** if possible). After training, build a **new** FlexIndex with that checkpoint (same `iter_flex_docs` as BM25) and plug it in with `retriever_for_loaded_encoder`. |

### Fine-tuning protocol (verbosity)

- **Unified:** `verbosity="all"` → more (query, doc+) pairs per judgment; one model; evaluate on Title / Description / Narrative queries as usual.  
- **Specialist:** `verbosity="title"` (or `description` / `narrative`) → separate checkpoints; each needs its **own** saved model directory **and** dense index because query and document towers share weights.

### Snapshot: BM25 vs DPR (MiniLM) from this notebook

Values below are taken from **embedded `pt.Experiment` output** in this notebook (dense model **`sentence-transformers/all-MiniLM-L6-v2`**, same title+abstract field as BM25). Re-run the **first DPR comparison block** (BM25 … DPR all-MiniLM) after changing code or data to refresh metrics.

| Query formulation | BM25 MAP | DPR MAP | BM25 nDCG@10 | DPR nDCG@10 |
|-------------------|----------|---------|--------------|-------------|
| Title             | 0.207   | 0.075   | 0.602        | 0.299       |
| Description       | 0.222   | 0.120   | 0.623        | 0.449       |
| Narrative         | 0.158   | 0.112   | 0.519        | 0.368       |

**Readout:** lexical **BM25** is stronger on MAP in this setup; **DPR** improves with **richer queries** (description vs title) but remains below BM25 on MAP here—consistent with domain vocabulary and indexing only title+abstract. **Fine-tuning** (`dpr_finetune.py`) is the intended next step to close that gap without replacing the sparse baseline.

### Optional: training-pair scale and CLI

`dpr_finetune.build_training_pairs` needs the corpus, a topics table (`qid`, title, description, narrative), and qrels with a `label` column. A **full** run uses all judgments with `label >= 1` and can yield on the order of **many tens of thousands** of (query, document) pairs when `verbosity="all"`.

Sanity-check without the notebook, from the `src/` directory:

`python dpr_finetune.py --pairs-smoke`

Full training (slow on CPU; GPU recommended):

`python dpr_finetune.py --train --verbosity all`


### Fine-tuning outside the notebook

Full **fine-tuning and re-indexing** are intentionally **not** run here (heavy on CPU; use a **GPU** when possible). After `python dpr_finetune.py --train`, build a **FlexIndex** with the same `iter_flex_docs` + `doc_encoder >> indexer` workflow as in `dpr_dense.py`, then use **`retriever_for_loaded_encoder`** on that directory so `pt.Experiment` sees **`docno`** in rankings.


# BioDPR

In [17]:
!pip install -q --upgrade torch torchvision torchaudio

  You can safely remove it manually.
  You can safely remove it manually.

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [40]:
import sys
from pathlib import Path

for _p in (Path.cwd(), Path.cwd() / "src"):
    if (_p / "dpr_dense.py").is_file():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break

from dpr_dense import make_biodpr_pipeline

biodpr_index_dir = Path("index/trec_covid_biodpr_complete").resolve()
biodpr_pipe = make_biodpr_pipeline(biodpr_index_dir)
if biodpr_pipe is None:
    print(
        f"No BioDPR FlexIndex at {biodpr_index_dir} (need pt_meta.json). "
        "Skipping BioDPR / hybrid until you add or build that index."
    )
else:
    print("BioDPR pipeline ready.")

Found BioDPR index! Loading...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: pritamdeka/S-PubMedBert-MS-MARCO
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BioDPR Pipeline ready!


In [41]:
all_models = [bm25, ql_dir, ql_jm, rm3_pipe, dpr_pipe]
all_model_names = ["BM25", "QL (Dir)", "QL (JM)", "BM25 + RM3", "DPR"]
if biodpr_pipe is not None:
    all_models.append(biodpr_pipe)
    all_model_names.append("BioDPR")

for variant_name, topics_df in query_variants.items():
    print(f"==================================================")
    print(f"Results for {variant_name.upper()} Queries:")
    print(f"==================================================")
    
    experiment_results = pt.Experiment(
        retr_systems=all_models,
        topics=topics_df,
        qrels=dataset.get_qrels(),
        eval_metrics=eval_metrics,
        names=all_model_names,
        baseline=0 
    )
    
    display(experiment_results)
    print("\n")

Results for TITLE Queries:


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.207121,0.807094,0.672,0.106150,0.601847,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.175731,0.684478,0.540,0.094298,0.444853,14.0,36.0,3.445424e-04,5.0,...,0.011665,8.0,23.0,7.844063e-04,16.0,33.0,5.811970e-03,8.0,40.0,4.739597e-07
2,QL (JM),0.128675,0.657621,0.452,0.075799,0.389103,2.0,48.0,2.471805e-09,7.0,...,0.018710,5.0,30.0,2.983346e-06,5.0,44.0,4.825223e-10,8.0,40.0,1.121550e-08
3,BM25 + RM3,0.215513,0.777866,0.684,0.109137,0.580049,30.0,20.0,1.205768e-01,4.0,...,0.202878,13.0,12.0,6.952988e-01,29.0,18.0,2.391932e-01,21.0,25.0,2.834572e-01
4,DPR,0.075122,0.563015,0.340,0.050606,0.299373,3.0,47.0,2.412138e-09,6.0,...,0.000482,3.0,34.0,1.268344e-08,5.0,43.0,1.140979e-08,6.0,43.0,2.527973e-09
5,BioDPR,0.100963,0.553502,0.420,0.068487,0.348634,8.0,42.0,3.971513e-07,4.0,...,0.000041,7.0,32.0,2.960906e-06,5.0,44.0,1.265811e-06,9.0,40.0,1.406583e-07




Results for DESCRIPTION Queries:


...

,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.222260,0.874500,0.716,0.112934,0.623256,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.185737,0.757972,0.592,0.101559,0.505937,9.0,41.0,2.067988e-05,7.0,...,0.037531,7.0,23.0,0.001941,16.0,29.0,2.971897e-03,15.0,35.0,0.000169
2,QL (JM),0.125899,0.715361,0.528,0.078242,0.444361,1.0,49.0,1.163376e-11,5.0,...,0.006109,6.0,30.0,0.000172,5.0,44.0,2.482803e-09,12.0,38.0,0.000012
3,BM25 + RM3,0.237755,0.842557,0.752,0.117835,0.655691,32.0,18.0,4.463815e-02,2.0,...,0.038771,13.0,10.0,0.192114,27.0,16.0,2.018099e-01,26.0,20.0,0.126493
4,DPR,0.119940,0.680775,0.524,0.075400,0.448718,6.0,44.0,2.223430e-09,5.0,...,0.004876,9.0,29.0,0.000935,8.0,41.0,1.290954e-06,12.0,38.0,0.000474
5,BioDPR,0.168095,0.808270,0.652,0.096608,0.597648,14.0,36.0,7.343930e-04,6.0,...,0.204821,18.0,24.0,0.194325,15.0,32.0,3.710228e-02,24.0,26.0,0.544790




Results for NARRATIVE Queries:


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.157843,0.718762,0.608,0.089073,0.518681,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.105462,0.556934,0.452,0.069563,0.347096,8.0,42.0,4.518420e-07,7.0,...,0.011115,10.0,26.0,0.003339,12.0,38.0,0.000012,11.0,35.0,0.000005
2,QL (JM),0.117736,0.638829,0.440,0.072307,0.388177,7.0,43.0,1.239431e-05,8.0,...,0.194593,6.0,25.0,0.000344,10.0,38.0,0.000231,11.0,33.0,0.000070
3,BM25 + RM3,0.175874,0.678626,0.560,0.090549,0.489098,19.0,31.0,1.503040e-01,4.0,...,0.101944,7.0,16.0,0.122430,17.0,30.0,0.703406,17.0,27.0,0.135629
4,DPR,0.111529,0.542542,0.424,0.069849,0.368355,14.0,36.0,2.405376e-03,9.0,...,0.023023,9.0,28.0,0.003070,13.0,36.0,0.004678,11.0,35.0,0.000409
5,BioDPR,0.140223,0.786781,0.600,0.089901,0.551742,20.0,30.0,2.476971e-01,12.0,...,0.245488,17.0,18.0,0.878472,21.0,28.0,0.903510,25.0,23.0,0.409129


# Hybrid Approach

In [42]:
import pyterrier as pt

def apply_rrf(res, k=60):
    res = res.copy()
    res['score'] = 1.0 / (k + res['rank'] + 1)
    return res

rrf_transformer = pt.apply.generic(apply_rrf)
bm25_rrf = bm25 >> rrf_transformer

if biodpr_pipe is not None:
    biodpr_rrf = biodpr_pipe >> rrf_transformer
    hybrid_pipe = bm25_rrf + biodpr_rrf
    all_models = [bm25, ql_dir, ql_jm, rm3_pipe, dpr_pipe, biodpr_pipe, hybrid_pipe]
    all_model_names = [
        "BM25",
        "QL (Dir)",
        "QL (JM)",
        "BM25 + RM3",
        "DPR",
        "BioDPR",
        "Hybrid (BM25 + BioDPR RRF)",
    ]
else:
    hybrid_pipe = None
    all_models = [bm25, ql_dir, ql_jm, rm3_pipe, dpr_pipe]
    all_model_names = ["BM25", "QL (Dir)", "QL (JM)", "BM25 + RM3", "DPR"]

for variant_name, topics_df in query_variants.items():
    print(f"==================================================")
    print(f"Results for {variant_name.upper()} Queries:")
    print(f"==================================================")
    
    experiment_results = pt.Experiment(
        retr_systems=all_models,
        topics=topics_df,
        qrels=dataset.get_qrels(),
        eval_metrics=eval_metrics,
        names=all_model_names,
        baseline=0
    )
    
    display(experiment_results)
    print("\n")

Results for TITLE Queries:


c:\Users\zosia\anaconda3\Lib\site-packages\pyterrier\_evaluation\_validation.py:76: UserWarning: Experiment Pipeline Validation Report

The following pipelines could not be validated (i.e., it is unclear what outputs they produce):
 - Pipeline #6: Hybrid (BM25 + BioDPR RRF) (<pyterrier._ops.Sum object at 0x000001F99FE90740>)
If these pipelines work, set validate='ignore' to remove this warning, or make them inspectable to clarify how they work.

See https://pyterrier.readthedocs.io/en/latest/troubleshooting/inspection.html for more information.
  warn(message)


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

...

c:\Users\zosia\anaconda3\Lib\site-packages\pyterrier\_evaluation\_validation.py:76: UserWarning: Experiment Pipeline Validation Report

The following pipelines could not be validated (i.e., it is unclear what outputs they produce):
 - Pipeline #6: Hybrid (BM25 + BioDPR RRF) (<pyterrier._ops.Sum object at 0x000001F99FE90740>)
If these pipelines work, set validate='ignore' to remove this warning, or make them inspectable to clarify how they work.

See https://pyterrier.readthedocs.io/en/latest/troubleshooting/inspection.html for more information.
  warn(message)


NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

NumpyRetriever scoring:   0%|          | 0/47 [00:00<?, ?docbatch/s]

,name,map,recip_rank,P_5,recall_100,ndcg_cut_10,map +,map -,map p-value,recip_rank +,...,recip_rank p-value,P_5 +,P_5 -,P_5 p-value,recall_100 +,recall_100 -,recall_100 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,BM25,0.157843,0.718762,0.608,0.089073,0.518681,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,QL (Dir),0.105462,0.556934,0.452,0.069563,0.347096,8.0,42.0,4.518420e-07,7.0,...,0.011115,10.0,26.0,0.003339,12.0,38.0,0.000012,11.0,35.0,0.000005
2,QL (JM),0.117736,0.638829,0.440,0.072307,0.388177,7.0,43.0,1.239431e-05,8.0,...,0.194593,6.0,25.0,0.000344,10.0,38.0,0.000231,11.0,33.0,0.000070
3,BM25 + RM3,0.175874,0.678626,0.560,0.090549,0.489098,19.0,31.0,1.503040e-01,4.0,...,0.101944,7.0,16.0,0.122430,17.0,30.0,0.703406,17.0,27.0,0.135629
4,DPR,0.111529,0.542542,0.424,0.069849,0.368355,14.0,36.0,2.405376e-03,9.0,...,0.023023,9.0,28.0,0.003070,13.0,36.0,0.004678,11.0,35.0,0.000409
5,BioDPR,0.140223,0.786781,0.600,0.089901,0.551742,20.0,30.0,2.476971e-01,12.0,...,0.245488,17.0,18.0,0.878472,21.0,28.0,0.903510,25.0,23.0,0.409129
6,Hybrid (BM25 + BioDPR RRF),0.207661,0.804306,0.712,0.104749,0.640086,41.0,9.0,1.232110e-07,13.0,...,0.143505,19.0,10.0,0.018952,36.0,12.0,0.000331,34.0,13.0,0.000133
